# 🧠 Observer Core — Colab Training Notebook

Train a specialized Observer Core model on free Colab T4 GPU.

**What this does:**
1. Clones the sovereign-edge-ai repo
2. Installs Unsloth + dependencies
3. Generates or downloads the training dataset
4. Fine-tunes your chosen model with QLoRA
5. Evaluates on test split
6. Saves adapter weights to Google Drive
7. Exports GGUF for phone deployment

**GPU:** Free T4 (15GB VRAM) — enough for all 10 models.
**Time:** ~5 min for quick test, ~2-4 hrs for full training.

## 0. Select Your Model

Change `MODEL_KEY` below to pick which model to train.

In [ ]:
# 🔧 CONFIGURATION — change these
MODEL_KEY = "qwen3.5-2b"          # Which model to train (see catalog below)
QUICK_MODE = True                  # True = 100 examples (5 min), False = full (2-4 hrs)

# Available models:
# 🥇 qwen3.5-2b     — 2.0B, ~450MB GGUF, newest, best balance
# 🥈 qwen3.5-0.8b   — 0.8B, ~180MB GGUF, lightest
# 📌 qwen3.5-4b     — 4.0B, ~900MB GGUF, higher quality
# 🥉 gemma4-e2b     — 2.0B, ~450MB GGUF, Google's latest
# 📌 ministral3-3b  — 3.0B, ~650MB GGUF, Mistral's newest
# 📌 deepseek-r1-1.5b — 1.5B, ~350MB GGUF, reasoning
# 📌 qwen3-1.7b     — 1.7B, ~380MB GGUF, previous gen
# 📌 smollm2-1.7b   — 1.7B, ~380MB GGUF, HuggingFace
# 📌 llama3.2-1b    — 1.2B, ~280MB GGUF, Meta
# 📌 qwen3-0.6b     — 0.6B, ~140MB GGUF, ultra-light

## 1. Install Dependencies

In [ ]:
!pip install -q unsloth transformers datasets accelerate peft bitsandbytes xformers trl
!pip install -q huggingface_hub

import torch
print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_mem/1e9:.1f} GB")

## 2. Clone Repo & Generate Dataset

In [ ]:
import os
if not os.path.exists("sovereign-edge-ai"):
    !git clone https://github.com/TheInnerI/sovereign-edge-ai.git

os.chdir("sovereign-edge-ai")
!pip install -q numpy scikit-learn  # data pipeline deps

# Generate dataset (quick: 1k, full: 11k)
n = 500 if QUICK_MODE else 5000
!python run_data_pipeline.py --quick 2>/dev/null || python -c "
from src.data.generator import ResidualDataGenerator, split_dataset, save_dataset
g = ResidualDataGenerator(seed=42)
d = g.generate_dataset(n_residuals=$n, n_coherence=200, n_contradictions=100, n_sequences=20, n_preferences=100, n_structured=100)
s = split_dataset(d)
save_dataset(s, d['metadata'])
print('Dataset ready!')
"

!python run_data_pipeline.py --stats

## 3. Mount Google Drive (for saving weights)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE_PATH = f"/content/drive/MyDrive/observer-core-models/{MODEL_KEY}"
!mkdir -p {DRIVE_PATH}
print(f"Weights will save to: {DRIVE_PATH}")

## 4. Define Observer Core System Prompt

In [ ]:
SYSTEM_PROMPT = """You are the Sovereign Edge Observer Core. Your ONLY functions are:
1. Detect residuals (gap between intent and outcome)
2. Score coherence (0.0-1.0) against six axioms
3. Detect contradictions with prior state
4. Propose minimal corrections (append-only)
5. Update invariant observer state (ψ₀)
6. Emit structured JSON output

Six Axioms (weighted):
- Awareness Is Law (20%): Observer is primary
- Truth Over Comfort (20%): Honest assessment; NO fabrication
- Coherence Over Features (15%): Internal consistency
- Append-Only Memory (15%): Corrections are additions
- Human Final Authority (15%): You propose; human decides
- Local Sovereignty (15%): Offline-capable

Output ONLY valid JSON. No markdown, no extra text."""

print("System prompt ready.")

## 5. Load & Format Training Data

In [ ]:
import json
from datasets import Dataset

def format_chat(example):
    inp = example.get("input", {})
    out = example.get("output", {})
    user = f"Intent: {inp.get('intent','')}\nPredicted: {inp.get('predicted','')}\nExecuted: {inp.get('executed','')}\nActual: {inp.get('actual','')}"
    assistant = json.dumps(out, ensure_ascii=False)
    text = f"<|im_start|>system\n{SYSTEM_PROMPT}<|im_end|>\n<|im_start|>user\n{user}<|im_end|>\n<|im_start|>assistant\n{assistant}<|im_end|>"
    return {"text": text}

examples = []
with open("data/datasets/observer-core/residuals_train.jsonl") as f:
    for line in f:
        examples.append(format_chat(json.loads(line)))

dataset = Dataset.from_list(examples)
print(f"Training examples: {len(dataset)}")
print(f"\nSample:\n{examples[0]['text'][:300]}...")

## 6. Load Model & Apply LoRA

In [ ]:
from unsloth import FastLanguageModel

# Model catalog — maps keys to HuggingFace model IDs
MODEL_MAP = {
    "qwen3.5-2b": "unsloth/Qwen3.5-2B-Instruct-bnb-4bit",
    "qwen3.5-0.8b": "unsloth/Qwen3.5-0.8B-Instruct-bnb-4bit",
    "qwen3.5-4b": "unsloth/Qwen3.5-4B-Instruct-bnb-4bit",
    "gemma4-e2b": "unsloth/gemma-4-E2B-it-unsloth-bnb-4bit",
    "ministral3-3b": "unsloth/Ministral-3-3B-Instruct-2512-unsloth-bnb-4bit",
    "deepseek-r1-1.5b": "unsloth/DeepSeek-R1-Distill-Qwen-1.5B-bnb-4bit",
    "qwen3-1.7b": "unsloth/Qwen3-1.7B-bnb-4bit",
    "smollm2-1.7b": "unsloth/SmolLM2-1.7B-Instruct-bnb-4bit",
    "llama3.2-1b": "unsloth/Llama-3.2-1B-Instruct-bnb-4bit",
    "qwen3-0.6b": "unsloth/Qwen3-0.6B-bnb-4bit",
}

model_id = MODEL_MAP.get(MODEL_KEY)
if not model_id:
    raise ValueError(f"Unknown model: {MODEL_KEY}. Choose from: {list(MODEL_MAP.keys())}")

print(f"Loading: {model_id}")
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=model_id,
    max_seq_length=2048,
    dtype=None,
    load_in_4bit=True,
)

# Apply LoRA
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"],
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=42,
)

print(f"Model loaded. LoRA applied.")
print(f"Trainable params: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

## 7. Train!

In [ ]:
from transformers import TrainingArguments
from trl import SFTTrainer
import time

epochs = 1 if QUICK_MODE else 3

training_args = TrainingArguments(
    output_dir="./output",
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    warmup_ratio=0.03,
    num_train_epochs=epochs,
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    fp16=not torch.cuda.is_bf16_supported(),
    bf16=torch.cuda.is_bf16_supported(),
    logging_steps=10,
    optim="adamw_8bit",
    weight_decay=0.01,
    seed=42,
    save_strategy="epoch",
    report_to="none",
)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    dataset_text_field="text",
    max_seq_length=2048,
    args=training_args,
)

print(f"Starting training: {epochs} epoch(s), {len(dataset)} examples...")
start = time.time()
trainer.train()
elapsed = time.time() - start
print(f"\n✅ Training complete in {elapsed/60:.1f} minutes!")

## 8. Save to Google Drive

In [ ]:
# Save LoRA adapter
adapter_path = f"{DRIVE_PATH}/adapter"
model.save_pretrained(adapter_path)
tokenizer.save_pretrained(adapter_path)

# Save system prompt
with open(f"{DRIVE_PATH}/OBSERVER_PROMPT.txt", "w") as f:
    f.write(SYSTEM_PROMPT)

# Save training metadata
import json
from datetime import datetime
meta = {
    "model_key": MODEL_KEY,
    "base_model": model_id,
    "training_examples": len(dataset),
    "epochs": epochs,
    "training_time_minutes": round(elapsed/60, 1),
    "trained_at": datetime.now().isoformat(),
}
with open(f"{DRIVE_PATH}/training_metadata.json", "w") as f:
    json.dump(meta, f, indent=2)

print(f"✅ Saved to {DRIVE_PATH}/")
print(f"   adapter/     — LoRA weights")
print(f"   OBSERVER_PROMPT.txt — system prompt")
print(f"   training_metadata.json — training info")
print(f"\n📥 Download to local machine:")
print(f"   Place adapter/ in: sovereign-edge-ai/output/observer-lora-{MODEL_KEY}/")
print(f"   Then run: python train_observer.py --eval --model {MODEL_KEY}")
print(f"   Or export: python train_observer.py --export --model {MODEL_KEY}")

## 9. Quick Eval (Optional)

In [ ]:
FastLanguageModel.for_inference(model)

# Load a few test examples
test_examples = []
with open("data/datasets/observer-core/residuals_test.jsonl") as f:
    for i, line in enumerate(f):
        if i >= 10: break
        test_examples.append(json.loads(line))

print(f"Running {len(test_examples)} test evaluations...\n")

json_ok = 0
for i, ex in enumerate(test_examples):
    inp = ex.get("input", {})
    expected = ex.get("output", {})
    expected_score = expected.get("coherence_score", 0.5)
    
    prompt = f"<|im_start|>system\n{SYSTEM_PROMPT}<|im_end|>\n"
    prompt += f"<|im_start|>user\nIntent: {inp.get('intent','')}\nPredicted: {inp.get('predicted','')}\nExecuted: {inp.get('executed','')}\nActual: {inp.get('actual','')}<|im_end|>\n<|im_start|>assistant\n"
    
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    outputs = model.generate(**inputs, max_new_tokens=256, temperature=0.1, do_sample=True, top_p=0.9)
    response = tokenizer.decode(outputs[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
    
    try:
        start = response.find("{")
        end = response.rfind("}")
        if start >= 0 and end > start:
            response = response[start:end+1]
        parsed = json.loads(response)
        score = parsed.get("coherence_score", "?")
        json_ok += 1
        match = "✅" if abs(score - expected_score) < 0.3 or (score == 0 and expected_score == 0) else "❌"
        print(f"  [{i+1}] expected={expected_score} got={score} JSON=OK {match}")
    except:
        print(f"  [{i+1}] JSON parse failed. Raw: {response[:80]}...")

print(f"\nJSON compliance: {json_ok}/{len(test_examples)} ({json_ok/len(test_examples)*100:.0f}%)")

## 10. Export GGUF (for phone deployment)

Requires llama.cpp. Run this cell after training to produce quantized GGUF files.

In [ ]:
# Install llama.cpp
!git clone -q https://github.com/ggerganov/llama.cpp /tmp/llama.cpp
!cd /tmp/llama.cpp && cmake -B build -DGGML_CUDA=OFF -q && cmake --build build -j2 -q 2>/dev/null

# Save merged model
merged_path = f"{DRIVE_PATH}/merged"
model.save_pretrained_merged(merged_path, tokenizer, save_method="merged_16bit")

# Convert to GGUF
!python /tmp/llama.cpp/convert_hf_to_gguf.py {merged_path} --outtype f16 --outfile {DRIVE_PATH}/observer-core-f16.gguf 2>/dev/null

# Quantize
!echo "Quantizing to IQ2_XS..."
!/tmp/llama.cpp/build/bin/llama-quantize {DRIVE_PATH}/observer-core-f16.gguf {DRIVE_PATH}/observer-core-IQ2_XS.gguf IQ2_XS 2>/dev/null

!echo "Quantizing to Q4_K_M..."
!/tmp/llama.cpp/build/bin/llama-quantize {DRIVE_PATH}/observer-core-f16.gguf {DRIVE_PATH}/observer-core-Q4_K_M.gguf Q4_K_M 2>/dev/null

import os
for f in os.listdir(DRIVE_PATH):
    if f.endswith('.gguf'):
        size_mb = os.path.getsize(f"{DRIVE_PATH}/{f}") / 1_048_576
        print(f"  {f}: {size_mb:.0f} MB")

print(f"\n✅ GGUFs saved to {DRIVE_PATH}/")
print(f"   Phone usage: ./llama-cli -m observer-core-IQ2_XS.gguf -f OBSERVER_PROMPT.txt")

---

## Done! 🎉

**Your trained model is in Google Drive:** `MyDrive/observer-core-models/{MODEL_KEY}/`

**To use locally:**
```bash
# Download adapter from Drive
# Place in: sovereign-edge-ai/output/observer-lora-{MODEL_KEY}/

# Evaluate
python train_observer.py --eval --model {MODEL_KEY}

# Or use GGUF on phone
./llama-cli -m observer-core-IQ2_XS.gguf -f OBSERVER_PROMPT.txt
```

**To train another model:** Change `MODEL_KEY` in cell 0 and Run All again.